# Joint (Mc, q) random-features mass model -- `tempest` run

This notebook tests `CHIMERA.population.mass.joint.random_features_joint_density`
-- a genuinely joint 2-D mass model, parametrized directly in (chirp mass
`Mc`, mass ratio `q`) rather than (m1, m2). Unlike every other mass model in
this package (`paired`: `p̃(m1) p̃(m2) f(m1,m2)/Z`; `conditioned`:
`p(m1) p(m2|m1)`), this one places a single joint random-features density
directly on `(Mc, q)` and pushes it forward to `(m1, m2)` through the
Jacobian of the change of variables:

    p(m1, m2) = p(Mc, q) * |d(Mc,q)/d(m1,m2)| = p(Mc, q) * Mc/m1^2

There is **no `beta`, no pairing function, no m2<=m1 mask to build in**:
every `(Mc, q)` in the rectangle `[mc_low, mc_high] x [q_low, 1]` maps to
exactly one valid `m2 <= m1` pair, so the ordering constraint that every
other model in this package has to construct by hand simply isn't there in
these coordinates.

Same overall structure as `random_features_blocks.ipynb` (its `tempest`-run
counterpart for the `paired` random-features model): scipy.stats priors,
flat `params_keys`/`prior_transform` vector, `tempest.Sampler`. **Section 0**
is fully local (no cluster data, no `tempest` needed) and additionally
includes a normalization sanity check specific to this model: `norm_2d` is a
finite `(Mc,q)`-grid estimate, and (as documented in
`random_features_joint_density`'s docstring, and found by direct
Monte-Carlo testing while building this model) a too-coarse grid can
under-count the true 2-D integral by a percent-level amount for a wiggly
enough draw -- worth checking whenever you widen the priors used here.

**Section 1** mirrors `bpl3p_run_tempest.py` for an actual `tempest`
hyperlikelihood run -- cluster-only, edit `dir_data`/`dir_chain` first.

In [ ]:
import os, sys
os.environ["CHIMERA_ENABLE_GPU"] = "True"
sys.path.append(os.getcwd() + '/../')

import numpy as np
import scipy.stats
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from CHIMERA.population.mass.joint import random_features_joint_density, pdf_joint_and_marg
from CHIMERA.population.mass import p_m1m2


## Config

`default_feature_scales`/`scale_prior_width`/`w_out_prior_scale` play the
exact same role as in `random_features_blocks.ipynb` (narrow, non-overlapping
lengthscale-block priors; `w_out`'s amplitude calibrated to `n_features` so
`g`'s std stays `O(1)` -- see that notebook's Config cell for the full
derivation, unchanged here).

`mc_grid_res`/`q_grid_res` control the resolution of the `(Mc,q)`
normalization grid (`random_features_joint_density.norm_2d`) -- raised
above their class default (800) here as a margin of safety, but
**deliberately not pushed higher, for a cost reason found while building
this notebook and demonstrated directly in Section 0.4**: `.update()`
recomputes this grid from scratch every call (`O(grid_res^2 * n_features)`
work), and `tempest` will call `.update()`/`log_likelihood` once per
particle *every step* -- with `n_particles` particles evaluated together
(likely batched/vectorized internally, the same way the 1-D `paired`
notebooks' jit/vmap checks assume), grid cost multiplies by particle count.
500x500 is a compromise for a first cluster run; if Section 1 is too slow
or runs out of memory, lower this (and/or `n_particles` below) first, and
use Section 0.3's Monte-Carlo check to confirm the lower resolution still
integrates close enough to 1.0 for your actual priors. If you widen the
priors below instead (wider `feature_scales`/`w_out_prior_scale`, i.e. a
wigglier model), you may need to *raise* it, and re-check the same way.

In [ ]:
n_features = 64                                    # keep modest (tens, not hundreds)
default_feature_scales = (1.0, 0.3, 0.1, 0.03)     # the 4 lengthscale "blocks"
n_scales = len(default_feature_scales)
scale_prior_width = 0.2                            # +/-20% narrow, non-overlapping log-uniform bands
target_std_g = 1.2                                 # desired std of g(x) itself -- see random_features_blocks.ipynb
w_out_prior_scale = target_std_g * (2.0 / n_features) ** 0.5

mc_grid_res, q_grid_res = 500, 500                 # (Mc,q) normalization-grid resolution -- see Config markdown; cost warning in 0.4

master_key = jax.random.PRNGKey(0)

# tempest / hyperlikelihood run settings (Section 1) -- modest defaults for a *test* run;
# bump these up for a production fit, same knobs as bpl3p_run_tempest.py's argparse flags.
n_particles = 256
n_steps = 5
n_total = 4096
prng_key = 0
resample = 'syst'

dir_data = '/leonardo_work/IscrC_MLGW/gwtc5/data/'
dir_chain = '/leonardo_work/IscrC_MLGW/gwtc5/res/tempest/'
chain_label = 'gwtc5_randomfeatures_joint_tempest'


## Priors

Same style as `random_features_blocks.ipynb`: `scipy.stats`-based
`uniform`/`loguniform`/`norm` helpers. `mass_priors` truncates the model in
its *native* `(Mc, q)` coordinates -- `mc_low`/`mc_high` (chirp-mass edges)
and `q_low` (mass-ratio lower edge; `q_high` is fixed at exactly `1.0`, not
sampled). There is no `beta` here: the joint model has no separate pairing
function, the mass-ratio dependence is folded directly into `p(Mc, q)`
itself. `w_out{i}` (one per feature) and `scale{i}` (one per block) are
re-assembled into the `w_out`/`feature_scales` arrays
`random_features_joint_density.update()` expects, exactly as in
`random_features_blocks.ipynb`.

In [ ]:
def uniform(low, high):
  return scipy.stats.uniform(loc=low, scale=high - low)

def loguniform(low, high):
  return scipy.stats.loguniform(a=low, b=high)

def norm(loc, scale):
  return scipy.stats.norm(loc=loc, scale=scale)

# --- mass-model (random_features_joint_density) priors -- native (Mc, q) coordinates ---
mass_priors = {
  "mc_low":  uniform(1.0, 6.0),     # chirp-mass lower edge
  "mc_high": uniform(20.0, 80.0),   # chirp-mass upper edge
  "q_low":   loguniform(0.02, 0.3), # mass-ratio lower edge; q_high fixed at 1.0
}

# narrow, non-overlapping priors on the 4 lengthscale blocks
scale_priors = {
  f"scale{i+1}": loguniform(s * (1. - scale_prior_width), s * (1. + scale_prior_width))
  for i, s in enumerate(default_feature_scales)
}

# one Normal prior per random-features output weight
w_out_priors = {f"w_out{i}": norm(loc=0., scale=w_out_prior_scale) for i in range(n_features)}

# --- cosmology / rate priors (same as bpl3p_run_tempest.py) ---
cosmo_rate_priors = {
  "H0": uniform(10., 200.),
  "gamma": uniform(0., 12.),
  "kappa": uniform(0., 6.),
  "zp": uniform(0., 4.),
}

priors = {**cosmo_rate_priors, **mass_priors, **scale_priors, **w_out_priors}
params_keys = list(priors.keys())
n_dim = len(params_keys)
prior_dists = [priors[k] for k in params_keys]

def prior_transform(u):
  return np.array([dist.ppf(u[i]) for i, dist in enumerate(prior_dists)])

print(f"n_dim = {n_dim}  ({n_features} w_out + {n_scales} feature_scales + "
      f"{len(mass_priors)} mass (Mc,q) + {len(cosmo_rate_priors)} cosmo/rate)")


## Section 0 -- local sanity checks (no cluster data / `tempest` needed)

### 0.1 Prior-predictive draws: marginal p(m1) and one joint p(m1, m2)

Draw `feature_scales`/`w_out` from the priors above, build a
`random_features_joint_density` realization for each draw, and overlay the
*marginal* `p(m1)` (obtained by numerically marginalizing the joint via
`pdf_joint_and_marg` -- there is no closed-form `p_m1` for this family, see
its docstring) -- same style of check as the other notebooks. Then, for one
representative draw, also show the actual *joint* `p(m1, m2)` as a 2-D
contour: unlike the 1-D marginal, this is the plot that actually shows what
"genuinely 2-D" buys you -- correlation structure between `m1` and `m2`
that a `paired`/`conditioned` model cannot represent, and the `m2 <= m1`
edge appearing automatically, with no explicit mask drawn by hand.

In [ ]:
n_draws = 10
mc_low_fixed, mc_high_fixed, q_low_fixed = 3.0, 45.0, 0.1  # fixed (Mc,q) box for this prior-predictive check

base_model = random_features_joint_density(
  mc_low=mc_low_fixed, mc_high=mc_high_fixed, q_low=q_low_fixed,
  n_features=n_features, feature_scales=default_feature_scales,
  mc_grid_res=mc_grid_res, q_grid_res=q_grid_res, key=master_key,
)

rng = np.random.default_rng(1)
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
drawn_scales = []
last_model = None
for i in range(n_draws):
  fs = jnp.array([scale_priors[f"scale{j+1}"].rvs(random_state=rng) for j in range(n_scales)])
  w_out = jnp.array([w_out_priors[f"w_out{j}"].rvs(random_state=rng) for j in range(n_features)])
  drawn_scales.append(np.asarray(fs))

  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  last_model = model_i
  out = pdf_joint_and_marg(model_i, res=(1500, 1500))
  ax.plot(out['m1'], out['p_m1_marg'], alpha=0.7, lw=1.2)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$  (marginal)')
ax.set_title(f'{n_draws} prior-predictive draws -- marginal p(m1)')
fig.tight_layout()
plt.show()

drawn_scales = np.array(drawn_scales)
print("drawn feature_scales -- min/max per block:")
for j in range(n_scales):
  print(f"  block {j+1} (default {default_feature_scales[j]:.3f}): "
        f"[{drawn_scales[:, j].min():.4f}, {drawn_scales[:, j].max():.4f}]")

# One representative draw, shown as the actual joint p(m1, m2)
out_last = pdf_joint_and_marg(last_model, res=(1200, 1200))
fig, ax = plt.subplots(1, 1, figsize=(6, 5.5))
cs = ax.contourf(out_last['m1mesh'], out_last['m2mesh'], out_last['p_joint'], levels=30, cmap='viridis')
ax.plot(out_last['m1'], out_last['m1'], 'w--', lw=1, alpha=0.7)  # m2 = m1 diagonal, for reference
ax.set_xlabel('$m_1$'); ax.set_ylabel('$m_2$')
ax.set_title('joint $p(m_1, m_2)$ for the last draw above\n(m2 <= m1 respected with no explicit mask)')
fig.colorbar(cs, ax=ax, label='$p(m_1, m_2)$')
fig.tight_layout()
plt.show()


### 0.2 Are the 4 blocks' priors actually narrow and non-overlapping?

Histogram many draws per block against the prior bounds -- identical check
to `random_features_blocks.ipynb`, since the block-lengthscale machinery is
shared code (`..paired.random_features`) and unaffected by the switch to a
2-D input.

In [ ]:
n_hist = 4000
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2), sharey=False)
for j, ax in enumerate(axes):
  draws = scale_priors[f"scale{j+1}"].rvs(size=n_hist, random_state=rng)
  ax.hist(draws, bins=40, color=f"C{j}", alpha=0.8)
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1, label='default')
  ax.set_title(f"block {j+1} prior")
  ax.set_xlabel("feature_scales[%d]" % j)
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


### 0.3 Normalization sanity check: is `p(m1, m2)` actually integrating to 1?

`random_features_joint_density.norm_2d` is a finite `(Mc,q)`-grid trapezoid
estimate (see its docstring): for a sufficiently wiggly draw, a too-coarse
grid can systematically under-count the true integral by a percent-level
amount, and -- unlike ordinary quadrature noise -- that bias does **not**
average out with more prior draws, only with a *finer* grid. This cell
checks it directly and independently of the model's own grid, via unbiased
Monte-Carlo integration of `p(m1, m2)` over its exact physical bounding box
(`_m1m2_plot_bounds`): if `mc_grid_res`/`q_grid_res` (Config cell) are fine
enough for your priors, the estimate below should sit within a few sigma of
1.0 for every draw checked.

In [ ]:
from CHIMERA.population.mass.joint.base import _m1m2_plot_bounds

def mc_normalization_check(model, n_mc=1_000_000, seed=0):
  m1_lo, m1_hi, m2_lo, m2_hi = (float(b) for b in _m1m2_plot_bounds(model))
  m1_lo, m2_lo, m1_hi, m2_hi = m1_lo * 0.98, m2_lo * 0.98, m1_hi * 1.02, m2_hi * 1.02
  area = (m1_hi - m1_lo) * (m2_hi - m2_lo)
  rng_mc = np.random.default_rng(seed)
  m1 = jnp.asarray(rng_mc.uniform(m1_lo, m1_hi, n_mc))
  m2 = jnp.asarray(rng_mc.uniform(m2_lo, m2_hi, n_mc))
  p = np.asarray(p_m1m2(model, m1, m2))
  mean, var = p.mean(), p.var()
  est, err = mean * area, np.sqrt(var / n_mc) * area
  return est, err

for i in range(5):
  fs = jnp.array([scale_priors[f"scale{j+1}"].rvs(random_state=rng) for j in range(n_scales)])
  w_out = jnp.array([w_out_priors[f"w_out{j}"].rvs(random_state=rng) for j in range(n_features)])
  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  est, err = mc_normalization_check(model_i)
  print(f"draw {i}: Z_grid={float(model_i.norm_2d):8.3f}   "
        f"MC integral of p(m1,m2) = {est:.4f} +/- {err:.4f}  "
        f"({(est - 1) / err:+.1f} sigma from 1.0)")


### 0.4 jit/vmap check: one compile, many `feature_scales`/`w_out` values -- and a real cost warning

Batch several particles with *different* `feature_scales` and `w_out`
through `.update()` + `p_m1m2` under a single `jax.jit` + `jax.vmap`, exactly
the way `tempest` will evaluate a particle population. The second call
reuses the trace (no recompilation) even though its `feature_scales` values
differ from the first call's.

**This is deliberately run at a *coarser* grid resolution than the Config
cell's `mc_grid_res`/`q_grid_res`** (via a dedicated `base_model_speed`).
`.update()` recomputes `norm_2d` from scratch every call -- `O(grid_res^2 *
n_features)` work -- and unlike the 1-D `paired` models (a 500-point 1-D
grid), `vmap`-ing that over a whole particle population multiplies grid
cost by particle count *in a single fused batched op*: `vmap`-ing this exact
cell at the Config cell's `mc_grid_res=q_grid_res=1000` over 64 particles
tried to allocate **32 GB** and crashed while building this notebook. This
is a genuine cost of a 2-D normalization grid, not a bug -- `tempest`
itself will hit the same cost every step, scaled by its actual
`n_particles`. If Section 1 is slow or runs out of memory on the cluster,
lower `mc_grid_res`/`q_grid_res` and/or `n_particles` first, and use
0.3's Monte-Carlo check to confirm the lower resolution is still accurate
enough for your priors.

In [ ]:
import time

n_particles_test = 64
fs_batch = jnp.stack([
  jnp.array([scale_priors[f"scale{j+1}"].rvs(random_state=rng) for j in range(n_scales)])
  for _ in range(n_particles_test)
])
w_out_batch = jnp.stack([
  jnp.array([w_out_priors[f"w_out{j}"].rvs(random_state=rng) for j in range(n_features)])
  for _ in range(n_particles_test)
])

m1_test = jnp.linspace(mc_low_fixed * 1.05, mc_high_fixed * 3.0, 4000)
m2_test = 0.7 * m1_test  # arbitrary fixed q=0.7 slice, just to exercise p_m1m2 pointwise

# Coarser grid dedicated to this speed check -- see the markdown above for why.
# (mc_grid_res/q_grid_res can only be set at fresh construction, not via .update() --
# they aren't in `.keys`, so a fresh instance is needed rather than `base_model.update(...)`.)
base_model_speed = random_features_joint_density(
  mc_low=mc_low_fixed, mc_high=mc_high_fixed, q_low=q_low_fixed,
  n_features=n_features, feature_scales=default_feature_scales,
  mc_grid_res=150, q_grid_res=150, key=master_key,
)

@jax.jit
def eval_particle(fs, w_out):
  model_i = base_model_speed.update(feature_scales=fs, w_out=w_out)
  return p_m1m2(model_i, m1_test, m2_test)

t0 = time.time()
out1 = jax.vmap(eval_particle)(fs_batch, w_out_batch)
out1.block_until_ready()
t1 = time.time()

fs_batch_2 = fs_batch * 1.05  # different concrete values, same shapes
out2 = jax.vmap(eval_particle)(fs_batch_2, w_out_batch)
out2.block_until_ready()
t2 = time.time()

print(f"first call (compile + run): {t1 - t0:.3f}s")
print(f"second call, different feature_scales values (should be much faster -- no retrace): {t2 - t1:.3f}s")
assert not jnp.any(jnp.isnan(out1)) and not jnp.any(jnp.isnan(out2))
print("no NaNs -- vmap over distinct feature_scales/w_out works with a single jit trace.")


## Section 1 -- full `tempest` hyperlikelihood run (cluster only)

Mirrors `examples/bpl3p_run_tempest.py` / `examples/random_features_blocks.ipynb`
exactly in structure: load GWTC5 PE + injection data, build
`cosmo`/`mass`/`rate` -> `population` -> `hyperlikelihood`, define
`log_likelihood`, run `tempest.Sampler`, save posterior + blobs to
`arviz.InferenceData`.

Needs `dir_data`/`dir_chain` (set in the config cell above) to point at real
GWTC5 files, and the `tempest` package (both cluster-only) -- **edit the
paths before running this section**.

In [ ]:
from CHIMERA import data
from CHIMERA.cosmo import flrw
from CHIMERA.rate import madau_dickinson
from CHIMERA import population as population_module, selection_function, hyperlikelihood
import h5py
import tempest as tp

## Load PE data
file_ev = dir_data + 'GWTC5_242CBC_FAR0.25_PE2048.h5'
pe_gw = data.load_gw_pe_samples(file_ev, parameters=['m1det', 'm2det', 'dL'], return_struct=True)
pe_prior = jnp.load(dir_data + 'GWTC5_242CBC_FAR0.25_PE2048prior.npy')
pe_gw = pe_gw.update(pe_prior=pe_prior)
Nev = len(pe_gw.dL)

## Load injection data and define selection function
file_inj = dir_data + "GWTC5_injections_far0.25.h5"
f_inj = h5py.File(file_inj)
Ngen = float(f_inj.attrs['Ngen'])
Tobs = float(f_inj.attrs['Tobs'])
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr': 'snr', 'log_pdraw': 'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)


In [ ]:
cosmo = flrw(H0=67.9, Om0=0.3065, z_max=5.)
mass = random_features_joint_density(
  n_features=n_features, feature_scales=default_feature_scales,
  mc_low=3.0, mc_high=45.0, q_low=0.1,
  mc_grid_res=mc_grid_res, q_grid_res=q_grid_res, key=master_key,
)
rate = madau_dickinson()
population = population_module(cosmo, mass, rate, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
  theta_gw_det=pe_gw,
  population=population,
  z_grids_res=300,
  selection_function=sel_fcn,
  pe_neff=2.0,
  inj_neff=None,
  kind_kde='fft',
  kernel='epan',
  kde_bw=None,
)


Assembling `hyperparams` from the flat `tempest` particle vector:
`dict(zip(params_keys, params))` gives every scalar hyperparameter directly
(now including `mc_low`/`mc_high`/`q_low` instead of `m_low`/`m_high`/`beta`);
`w_out{i}`/`scale{i}` are popped out and re-stacked into the
`w_out`/`feature_scales` arrays `random_features_joint_density.update()`
(via `mass.keys`) expects -- exactly the pattern
`random_features_blocks.ipynb` uses for its `paired` counterpart.

In [ ]:
def log_likelihood(params):
  raw = dict(zip(params_keys, params))
  w_out = jnp.array([raw.pop(f"w_out{i}") for i in range(n_features)])
  feature_scales = jnp.array([raw.pop(f"scale{i+1}") for i in range(n_scales)])
  hyperparams = {**raw, "w_out": w_out, "feature_scales": feature_scales}
  log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
  return log_like, log_like_evs, N_exp, neff_inj

# Prior-predictive finite-likelihood check before launching the full sampler: with n_particles
# particles drawn from the prior, tempest needs enough of them finite to seed its initial
# population from. Keep the accepted draws around too, to inspect what they look like as p(m1).
n_check = 10000
accepted_thetas = []
for s in range(n_check):
  theta = prior_transform(np.random.default_rng(s).uniform(size=n_dim))
  ll, *_ = log_likelihood(theta)
  if np.isfinite(ll):
    accepted_thetas.append(theta)
print(f"{len(accepted_thetas)}/{n_check} prior draws gave a finite log-likelihood "
      f"(~{len(accepted_thetas)/n_check*n_particles:.0f} expected finite particles at init)")


### What does the finite-likelihood region of the prior look like as a mass function?

Before spending compute on the full run, plot the marginal `p(m1)` for a
handful of the `accepted_thetas` collected above -- these are draws that
passed every event's `pe_neff` check, so this is a sanity check on what the
sampler's *initial* population actually looks like, not yet the posterior.

In [ ]:
n_plot = min(15, len(accepted_thetas))
plot_idx = np.random.default_rng(0).choice(len(accepted_thetas), size=n_plot, replace=False)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for i in plot_idx:
  raw = dict(zip(params_keys, accepted_thetas[i]))
  w_out_i = jnp.array([raw.pop(f"w_out{k}") for k in range(n_features)])
  fs_i = jnp.array([raw.pop(f"scale{k+1}") for k in range(n_scales)])
  mass_i = mass.update(
    w_out=w_out_i, feature_scales=fs_i,
    mc_low=raw["mc_low"], mc_high=raw["mc_high"], q_low=raw["q_low"],
  )
  out = pdf_joint_and_marg(mass_i, res=(1200, 1200))
  ax.plot(out['m1'], out['p_m1_marg'], alpha=0.6, lw=1)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$m_1$"); ax.set_ylabel("$p(m_1)$  (marginal)")
ax.set_title(f"{n_plot} accepted (finite-likelihood) prior draws")
fig.tight_layout()
plt.show()


In [ ]:
sampler = tp.Sampler(
  prior_transform=prior_transform,
  log_likelihood=log_likelihood,
  n_dim=n_dim,
  n_particles=n_particles,
  n_steps=n_steps,
  resample=resample,
  random_state=prng_key,
  output_dir=dir_chain,
  output_label=chain_label,
)

sampler.run(n_total=n_total, save_every=1, resume_state_path=None)

samples, weights, logl, blobs = sampler.posterior(resample=True, return_blobs=True)
samples_dict = {k: samples[:, i] for i, k in enumerate(priors)}

log_like_evs_blob = blobs[:, 0]
N_exp_blob = blobs[:, 1]
neff_inj_blob = blobs[:, 2]

logZ, logZerr = sampler.evidence()
print(f"logZ = {logZ}  logZerr = {logZerr}")


In [ ]:
import xarray as xr
import arviz as az

n_chains = 1
n_draws_post = len(samples)
n_events = np.stack(log_like_evs_blob).shape[-1]

posterior_ds = xr.Dataset(
  {k: (['chain', 'draw'], v.reshape(n_chains, n_draws_post)) for k, v in samples_dict.items()}
)
sample_stats_ds = xr.Dataset({
  'log_likelihood': (['chain', 'draw'], logl.reshape(n_chains, n_draws_post)),
  'weights': (['chain', 'draw'], weights.reshape(n_chains, n_draws_post)),
  'log_like_evs': (['chain', 'draw', 'event'], np.stack(log_like_evs_blob).reshape(n_chains, n_draws_post, n_events)),
  'N_exp': (['chain', 'draw'], np.asarray(N_exp_blob).reshape(n_chains, n_draws_post)),
  'Neff_inj': (['chain', 'draw'], np.asarray(neff_inj_blob).reshape(n_chains, n_draws_post)),
})

idata = az.InferenceData(posterior=posterior_ds, sample_stats=sample_stats_ds)
idata.attrs['log_evidence'] = float(logZ)
idata.attrs['log_evidence_err'] = str(logZerr) if logZerr is None else float(logZerr)
idata.to_netcdf(f"{dir_chain}/{chain_label}_final.nc")


## Section 2 -- inspecting the 4 blocks and the joint shape in the posterior

Compares the posterior over the 4 `feature_scales` against their (narrow)
priors, same diagnostic as `random_features_blocks.ipynb`. Also draws a
posterior-predictive marginal `p(m1)` band, and -- since this model's whole
point is the joint structure -- the posterior-predictive **joint**
`p(m1, m2)` at the posterior median hyperparameters, which a
`paired`/`conditioned` model has no equivalent of.

In [ ]:
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2))
for j, ax in enumerate(axes):
  post = samples_dict[f"scale{j+1}"]
  prior_draws = scale_priors[f"scale{j+1}"].rvs(size=5000, random_state=np.random.default_rng(j))
  ax.hist(prior_draws, bins=40, density=True, alpha=0.4, label='prior', color='gray')
  ax.hist(post, bins=40, density=True, alpha=0.7, label='posterior', color=f"C{j}")
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1)
  ax.set_title(f"block {j+1}")
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
n_ppc = 150
idx = np.random.default_rng(0).choice(len(samples), size=n_ppc, replace=False)

curves, m1_ref = [], None
for i in idx:
  w_out_i = jnp.array([samples_dict[f"w_out{k}"][i] for k in range(n_features)])
  fs_i = jnp.array([samples_dict[f"scale{k+1}"][i] for k in range(n_scales)])
  model_i = mass.update(
    w_out=w_out_i, feature_scales=fs_i,
    mc_low=samples_dict['mc_low'][i], mc_high=samples_dict['mc_high'][i], q_low=samples_dict['q_low'][i],
  )
  out = pdf_joint_and_marg(model_i, res=(1200, 1200))
  if m1_ref is None:
    m1_ref = out['m1']
  curves.append(jnp.interp(m1_ref, out['m1'], out['p_m1_marg']))

curves = jnp.stack(curves)
lo, med, hi = jnp.percentile(curves, jnp.array([5., 50., 95.]), axis=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.fill_between(m1_ref, lo, hi, alpha=0.3, label='90% posterior-predictive')
ax.plot(m1_ref, med, lw=1.5, label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$  (marginal)')
ax.legend()
fig.tight_layout()
plt.show()

# Posterior-median joint p(m1, m2) -- the genuinely-2D payoff of this model.
# "Representative" draw = the one whose whole marginal curve (in log-density) is
# closest overall to the median curve above, not just at a single grid point.
log_dist = jnp.sum((jnp.log(curves + 1e-300) - jnp.log(med + 1e-300)) ** 2, axis=1)
med_idx = idx[jnp.argmin(log_dist)]
w_out_med = jnp.array([samples_dict[f"w_out{k}"][med_idx] for k in range(n_features)])
fs_med = jnp.array([samples_dict[f"scale{k+1}"][med_idx] for k in range(n_scales)])
model_med = mass.update(
  w_out=w_out_med, feature_scales=fs_med,
  mc_low=samples_dict['mc_low'][med_idx], mc_high=samples_dict['mc_high'][med_idx],
  q_low=samples_dict['q_low'][med_idx],
)
out_med = pdf_joint_and_marg(model_med, res=(1200, 1200))
fig, ax = plt.subplots(1, 1, figsize=(6, 5.5))
cs = ax.contourf(out_med['m1mesh'], out_med['m2mesh'], out_med['p_joint'], levels=30, cmap='viridis')
ax.plot(out_med['m1'], out_med['m1'], 'w--', lw=1, alpha=0.7)
ax.set_xlabel('$m_1$'); ax.set_ylabel('$m_2$')
ax.set_title('posterior-predictive joint $p(m_1, m_2)$\n(one representative posterior draw)')
fig.colorbar(cs, ax=ax, label='$p(m_1, m_2)$')
fig.tight_layout()
plt.show()
